# Lab 2: Neural Network Classification using the Iris Dataset

In this lab, I use a neural network from scikit-learn to classify iris flowers into
their correct species. The dataset contains four input features and three possible
output classes.

Before training the neural network, I normalise the input features using min-max
normalisation and convert the target labels into one-hot encoded vectors.

The dataset is divided into training, validation and test sets. During training,
the sum-of-squares error is calculated for both the training and validation sets
after every epoch so that the learning progress can be monitored.

In [1]:
import numpy as np
import pandas as pd
import warnings

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from sklearn.exceptions import ConvergenceWarning

# Hide convergence warnings because we intentionally train one epoch at a time
warnings.filterwarnings("ignore", category=ConvergenceWarning)



## Loading the Dataset
The Iris dataset does not contain column headings,
so I manually assign the column names A, B, C, D and E.
Columns A to D contain the input features and
column E contains the target class.

In [2]:
# Load the dataset

df = pd.read_csv("Iris.csv", sep=";", header=None)

# Assign column names

df.columns = ["A", "B", "C", "D", "E"]

df.head()

,A,B,C,D,E
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


## Separating Inputs and Targets

The first four columns are used as inputs while the last column contains the
target classes.

In [3]:
# Inputs (A-D)
X = df[["A", "B", "C", "D"]].values

# Target (E)
y = df["E"].values

print("First 5 Input Samples:")
print(X[:5])

print("\nFirst 5 Target Values:")
for target in y[:5]:
    print(target)

First 5 Input Samples:
[[5.1 3.5 1.4 0.2]
 [4.9 3.  1.4 0.2]
 [4.7 3.2 1.3 0.2]
 [4.6 3.1 1.5 0.2]
 [5.  3.6 1.4 0.2]]

First 5 Target Values:
Iris-setosa
Iris-setosa
Iris-setosa
Iris-setosa
Iris-setosa


## Min-Max Normalisation

The input features are scaled to the range [0,1] using min-max normalisation.
This helps the neural network train more efficiently because all features are
on a similar scale.

In [4]:
scaler = MinMaxScaler()

X = scaler.fit_transform(X)

print(X[:5])

[[0.22222222 0.625      0.06779661 0.04166667]
 [0.16666667 0.41666667 0.06779661 0.04166667]
 [0.11111111 0.5        0.05084746 0.04166667]
 [0.08333333 0.45833333 0.08474576 0.04166667]
 [0.19444444 0.66666667 0.06779661 0.04166667]]


## One-Hot Encoding

The target labels are converted into one-hot encoded vectors. This allows the
network to represent the three output classes using three output neurons.

In [5]:
encoder = OneHotEncoder(sparse_output=False)

y_onehot = encoder.fit_transform(y.reshape(-1, 1))

print(y_onehot[:5])

[[1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]]


## Splitting the Dataset

The data is divided into:

- 50% Training
- 25% Validation
- 25% Testing



In [6]:
# First split:
# 50% training and 50% temporary

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y_onehot,
    test_size=0.5,
    random_state=42,
    stratify=y
)

# Split temporary set into validation and test

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=42
)

print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Test samples:", len(X_test))

Training samples: 75
Validation samples: 37
Test samples: 38


## Creating the Neural Network

The network consists of:

- 4 input nodes
- 1 hidden layer with 20 neurons
- 3 output nodes

The model is trained one epoch at a time using warm_start=True. This allows the
training and validation errors to be displayed after each epoch.

In [7]:
mlp = MLPClassifier(
    hidden_layer_sizes=(20,),
    activation="relu",
    solver="sgd",
    learning_rate_init=0.05,
    max_iter=1,
    warm_start=True,
    random_state=42
)

## Training the Neural Network

After every epoch, the sum-of-squares error (SSE) is calculated for both the
training and validation datasets.

In [10]:
epochs = 100

train_labels = np.argmax(y_train, axis=1)

for epoch in range(epochs):

    mlp.fit(X_train, train_labels)

    train_output = mlp.predict_proba(X_train)
    val_output = mlp.predict_proba(X_val)

    train_sse = np.sum((y_train - train_output) ** 2)
    val_sse = np.sum((y_val - val_output) ** 2)

    print(
        f"Epoch {epoch+1:3d} | "
        f"Training SSE = {train_sse:.4f} | "
        f"Validation SSE = {val_sse:.4f}"
    )

Epoch   1 | Training SSE = 21.2055 | Validation SSE = 10.9104
Epoch   2 | Training SSE = 21.0437 | Validation SSE = 10.8370
Epoch   3 | Training SSE = 20.8849 | Validation SSE = 10.7655
Epoch   4 | Training SSE = 20.7284 | Validation SSE = 10.6955
Epoch   5 | Training SSE = 20.5736 | Validation SSE = 10.6261
Epoch   6 | Training SSE = 20.4207 | Validation SSE = 10.5564
Epoch   7 | Training SSE = 20.2718 | Validation SSE = 10.4880
Epoch   8 | Training SSE = 20.1262 | Validation SSE = 10.4216
Epoch   9 | Training SSE = 19.9838 | Validation SSE = 10.3560
Epoch  10 | Training SSE = 19.8434 | Validation SSE = 10.2915
Epoch  11 | Training SSE = 19.7050 | Validation SSE = 10.2279
Epoch  12 | Training SSE = 19.5666 | Validation SSE = 10.1651
Epoch  13 | Training SSE = 19.4329 | Validation SSE = 10.1046
Epoch  14 | Training SSE = 19.3003 | Validation SSE = 10.0452
Epoch  15 | Training SSE = 19.1704 | Validation SSE = 9.9863
Epoch  16 | Training SSE = 19.0407 | Validation SSE = 9.9272
Epoch  17 

## Evaluating the Model

After training is complete, the neural network is tested on the unseen test
dataset. The classification accuracy is then calculated.

In [11]:
y_test_labels = np.argmax(y_test, axis=1)

y_pred = mlp.predict(X_test)

accuracy = accuracy_score(y_test_labels, y_pred)

print(f"Test Accuracy: {accuracy * 100:.2f}%")

Test Accuracy: 97.37%
